# 工具的应用案例
## 1、args_schema使用
举例：

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

In [3]:
#定义工具
from pydantic import BaseModel,Field
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools.convert import tool
class   WeatherSchema(BaseModel):
    city: str = Field(default="北京",description="具体的城市名称")
    if_forecast: bool = Field(
        default=False,
        description="是否包含明日天气"
    )

@tool("get_weather_and_forecast",description="查询当日的天气。可以包含明天的天气预报",args_schema=WeatherSchema)
def get_weather(city:str,if_forecast:bool):
    res = f"{city}今天天气不错"
    if if_forecast:
        res +="\n明天下雨"
    return res

print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日的天气。可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '具体的城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明日天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [6]:
from langchain_core.messages import HumanMessage
#将工具绑定在模型上
model_with_tool=model.bind_tools([get_weather])

#2、维护一个消息列表
messages=[HumanMessage("今天杭州的天气怎么样")]
#3、调用模型
response= model_with_tool.invoke(messages)
messages.append(response)

#4、获取相应中的tool_calls字段信息
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        #5、调用工具，（因为大模型不能主动调用工具，所以我们主动让工具调用执行）
         tool_message = get_weather.invoke(tool_call)
messages.append(tool_message)

#6、调用模型,得到AImessage
final_response = model.invoke(messages)
#7、添加到消息列表中
messages.append(final_response)
#遍历一下消息列表
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州的天气怎么样
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_00_WyH9cDn9xvwquDjDsc699439)
 Call ID: call_00_WyH9cDn9xvwquDjDsc699439
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州今天天气不错
================================== Ai Message ==================================

查询结果显示，杭州今天天气不错，以晴到多云为主。

不过这个信息比较笼统，杭州这个季节天气多变，早晚温差也比较大。为了让你准确掌握穿衣和出行安排，建议你通过手机上的天气App（如中国天气通、墨迹天气）或网页搜索“杭州天气”，获取实时的温度、湿度和降雨预报信息。


## 2、docstring

In [8]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

In [9]:
#定义工具
from pydantic import BaseModel,Field
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools.convert import tool
class   WeatherSchema(BaseModel):
    city: str = Field(default="北京",description="具体的城市名称")
    if_forecast: bool = Field(
        default=False,
        description="是否包含明日天气"
    )

@tool("get_weather_and_forecast",parse_docstring=True)
def get_weather(city:str="北京",if_forecast:bool=False):
    """
    查询当日的天气，可以包含明天的天气预报

    Args:
        city:城市名称
        if_forecast:是否包含明天的天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res +="\n明天下雨"
    return res

print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日的天气，可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明天的天气', 'type': 'boolean'}}, 'type': 'object'}}}
